# Question 2

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
from scipy.stats import chi2 as scipy_chi2

from a6cw.shear import (
    BIN_EDGES,
    BIN_CENTRES,
    COLOURS,
    LABELS,
    load_positions,
    compute_tangential_ellipticities,
    mean_tangential_shear,
    estimate_shape_noise_variance,
    optimal_weights,
    plot_tangential_shear_profiles,
    plot_cross_shear_null_test,
    build_nfw_theory,
    log_likelihood,
    compute_posterior_grid,
    plot_joint_posterior,
    plot_mass_posterior_fixed_zs,
    plot_best_fit_overlay,
)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

HALO_POS_FILE = DATA_DIR / "halo_positions.txt"
SOURCE_POS_FILE = DATA_DIR / "source_positions.txt"

# Q1 ellipticity outputs
ELLIP_FILES = {
    "unweighted": OUTPUT_DIR / "ellipticities_unweighted.npy",
    "weighted": OUTPUT_DIR / "ellipticities_weighted.npy",
    "hsm": OUTPUT_DIR / "ellipticities_hsm.npy",
}

# Simulation truth parameters (fixed throughout Q2)
HALO_REDSHIFT = 0.3
CONCENTRATION = 4.0
OMEGA_M = 0.3
OMEGA_LAM = 0.7

In [2]:
def main():
    # ------------------------------------------------------------------
    # Load positions and Q1 ellipticities
    # ------------------------------------------------------------------
    halo_pos   = load_positions(HALO_POS_FILE)
    source_pos = load_positions(SOURCE_POS_FILE)
    print(f'Loaded {len(halo_pos)} halos and {len(source_pos)} sources')

    ellipticities = {
        name: np.load(path) for name, path in ELLIP_FILES.items()
    }

    # ------------------------------------------------------------------
    # (a) + (b)  Tangential shear profiles for all three methods
    # ------------------------------------------------------------------
    profiles = {}
    for method, e_arr in ellipticities.items():
        e1 = e_arr[:, 0]
        e2 = e_arr[:, 1]

        print(f'\nMethod: {method}')
        r_pairs, et_pairs, ex_pairs = compute_tangential_ellipticities(
            halo_pos, source_pos, e1, e2
        )

        # (a) Simple (unweighted) mean
        gt_s, gt_err_s, n_s = mean_tangential_shear(
            r_pairs, et_pairs, BIN_EDGES, weights=None
        )

        # (b) Optimal (inverse-variance) weighted mean
        w = optimal_weights(et_pairs)
        gt_o, gt_err_o, n_o = mean_tangential_shear(
            r_pairs, et_pairs, BIN_EDGES, weights=w
        )

        # Cross-shear (B-mode) null test — same binning, no weighting needed
        gx_s, gx_err_s, _ = mean_tangential_shear(
            r_pairs, ex_pairs, BIN_EDGES, weights=None
        )

        sigma2 = estimate_shape_noise_variance(et_pairs)
        print(f'Shape noise variance sigma_e^2 = {sigma2:.5f}')
        print(f'Bin pair counts (simple): {n_s}')

        profiles[method] = {
            'gt_simple':     gt_s,
            'gt_err_simple': gt_err_s,
            'n_simple':      n_s,
            'gt_optimal':    gt_o,
            'gt_err_optimal':gt_err_o,
            'n_optimal':     n_o,
            # Aliases used by plot_tangential_shear_profiles
            'gt_opt':        gt_o,
            'gt_err_opt':    gt_err_o,
            # Cross-shear null test
            'gx_simple':     gx_s,
            'gx_err_simple': gx_err_s,
        }

    np.save(OUTPUT_DIR / 'tangential_shear_profiles.npy', profiles)

    # ------------------------------------------------------------------
    # (c) Plot: all three methods side by side
    # ------------------------------------------------------------------
    print('\nPlotting tangential shear profiles...')
    plot_tangential_shear_profiles(profiles, OUTPUT_DIR)

    print('Plotting cross-shear null test...')
    plot_cross_shear_null_test(profiles, OUTPUT_DIR)

    # ------------------------------------------------------------------
    # (d) Posterior inference using HSM (most reliable estimator)
    # ------------------------------------------------------------------
    print('\nRunning posterior grid for HSM estimator...')
    gt_hsm     = profiles['hsm']['gt_simple']
    gt_err_hsm = profiles['hsm']['gt_err_simple']

    mass_grid = np.logspace(13.8, 15.2, 1000)
    zs_grid   = np.linspace(0.31, 0.80, 1000)

    log_post = compute_posterior_grid(gt_hsm, gt_err_hsm, mass_grid, zs_grid)
    np.save(OUTPUT_DIR / 'q2d_log_posterior.npy', log_post)

    print('\nPlotting joint posterior with 1D marginals...')
    mass_map, zs_map, m_lo, m_hi, zs_lo, zs_hi = plot_joint_posterior(
        log_post, mass_grid, zs_grid, OUTPUT_DIR
    )

    # ------------------------------------------------------------------
    # Summary: joint MAP with 1D marginal errors
    # ------------------------------------------------------------------
    print('\n' + '='*60)
    print('PARAMETER ESTIMATES (1D marginal posteriors)')
    print('='*60)
    print(f'  Halo mass:       M = {mass_map/1e14:.3f}'
          f' +{(m_hi-mass_map)/1e14:.3f} -{(mass_map-m_lo)/1e14:.3f}'
          f'  x10^14 M_sun')
    print(f'  Source redshift: z_s = {zs_map:.3f}'
          f' +{zs_hi-zs_map:.3f} -{zs_map-zs_lo:.3f}')
    print(f'  (68% credible intervals from marginal posteriors)')
    print('='*60)

    print('\nPlotting mass posterior at fixed z_s = 0.6...')
    mass_map_06, mass_lo_06, mass_hi_06 = plot_mass_posterior_fixed_zs(
        log_post, mass_grid, zs_grid, fixed_zs=0.6, output_dir=OUTPUT_DIR
    )

    print('\n' + '='*60)
    print('MASS ESTIMATE AT FIXED z_s = 0.6')
    print('='*60)
    print(f'  M = {mass_map_06/1e14:.3f}'
          f' +{(mass_hi_06-mass_map_06)/1e14:.3f}'
          f' -{(mass_map_06-mass_lo_06)/1e14:.3f}'
          f'  x10^14 M_sun  (68% CI)')
    print('='*60)

    print('\nPlotting MAP theory overlay...')
    j_06 = np.argmin(np.abs(zs_grid - 0.6))
    mass_map_06_grid = mass_grid[np.argmax(log_post[:, j_06])]
    plot_best_fit_overlay(
        gt_hsm, gt_err_hsm, mass_map, zs_map, mass_map_06_grid, OUTPUT_DIR
    )

    print('\nQ2 complete.')


if __name__ == '__main__':
    main()


Loaded 100 halos and 100000 sources

Method: unweighted


Shape noise variance sigma_e^2 = 0.01465
Bin pair counts (simple): [   238    456    960   1940   4052   8238  16348  33205  66314 129764]

Method: weighted


Shape noise variance sigma_e^2 = 0.00013
Bin pair counts (simple): [   253    493   1047   2098   4354   8903  17723  35978  71727 140558]

Method: hsm


Shape noise variance sigma_e^2 = 0.00328
Bin pair counts (simple): [   253    493   1047   2099   4355   8905  17723  35979  71729 140562]

Plotting tangential shear profiles...


Saved outputs/q2c_tangential_shear_profiles.png
Plotting cross-shear null test...


Saved outputs/q2c_cross_shear_null_test.png

Running posterior grid for HSM estimator...



Plotting joint posterior with 1D marginals...
MAP: mass = 7.401e+14 M_sun,  z_s = 0.468
Marginal mass:  MAP = 6.380e+14  68% CI = [5.123e+14, 9.488e+14] M_sun
  -> 6.380 +3.108 -1.257  x10^14 M_sun
Marginal z_s:   MAP = 0.452  68% CI = [0.430, 0.563]
  -> 0.452 +0.110 -0.023


Saved outputs/q2d_joint_posterior.png

PARAMETER ESTIMATES (1D marginal posteriors)
  Halo mass:       M = 7.401 +2.087 -2.278  x10^14 M_sun
  Source redshift: z_s = 0.468 +0.095 -0.038
  (68% credible intervals from marginal posteriors)

Plotting mass posterior at fixed z_s = 0.6...
  Fixed z_s = 0.600:
    MAP mass = 4.650e+14 M_sun
    68% CI   = [4.488e+14, 4.849e+14] M_sun


  Saved outputs/q2d_mass_posterior_fixed_zs.png

MASS ESTIMATE AT FIXED z_s = 0.6
  M = 4.650 +0.199 -0.162  x10^14 M_sun  (68% CI)

Plotting MAP theory overlay...


Saved outputs/q2d_best_fit_overlay.png

Q2 complete.
